# 🎧 Teoría de Representaciones Espectrales para Audio
## STFT, Espectrogramas Mel y MFCCs — Aplicados a vocalizaciones de aves amazónicas

**Proyecto:** SelvaSonic — Clasificador bioacústico con CNN + Attention  
**Autores:** Laura Ruiz Arango & Jose Aldair Molina Méndez  
**Curso:** Aprendizaje Automático — Prof. Alcides Montoya  
**Universidad Nacional de Colombia — Sede Medellín, 2026**

---

### ¿Para qué este notebook?

Nuestro modelo SelvaSonic recibe **audio crudo** (una señal 1D en el tiempo) y debe clasificar la especie.
Pero las redes neuronales no entienden audio crudo directamente — necesitamos convertirlo en una
**representación 2D** (como una imagen) que capture la información relevante: **qué frecuencias suenan
y cuándo**.

Este notebook cubre los 3 conceptos fundamentales para esa conversión:

| # | Concepto | ¿Qué hace? | ¿Para qué lo usamos? |
|---|---------|-----------|--------------------|
| 1 | **STFT** | Descompone el audio en frecuencias a lo largo del tiempo | Base teórica de todo lo demás |
| 2 | **Espectrograma Mel** | Aplica escala perceptual al STFT | **Input principal del modelo CNN** |
| 3 | **MFCCs** | Comprime el Mel-spectrograma con DCT | Features complementarios opcionales |

La cadena completa es:

```
Audio crudo → STFT → |STFT|² → Mel filterbank → log() → Mel-spectrograma
                                                              ↓
                                                           DCT → MFCCs
```

Cada sección incluye: **intuición → matemáticas → visualización con audio real del proyecto**.

## 0. Setup: importaciones y carga de audio de prueba

In [ ]:
# === Importaciones ===
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path
import sys

# Importar nuestro módulo de carga de audio
sys.path.insert(0, str(Path.cwd().parent))  # para que src/ sea importable
from src.audio_io import load_audio, summarize, SAMPLE_RATE

# === Configuración de visualización ===
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# === Colores del proyecto (paleta amazónica) ===
COLORS = {
    'waveform': '#2E7D32',    # verde selva
    'spectrogram': 'magma',   # colormap para espectrogramas
    'mel': 'inferno',         # colormap para mel-spectrogramas
    'accent': '#FF6F00',      # naranja amazónico
}

print(f'✅ librosa {librosa.__version__}')
print(f'✅ Sample rate del proyecto: {SAMPLE_RATE} Hz')

### Cargar un audio real del dataset

Vamos a usar un audio de **Lipaugus vociferans** (Piha gritona) porque tiene un canto
muy distintivo y con estructura frecuencial clara — ideal para visualizar conceptos.

Si no tienes esa especie, el código busca cualquier audio disponible en `data/raw/`.

In [ ]:
# Intentar cargar Lipaugus vociferans (Piha gritona) — canto muy distintivo
data_dir = Path.cwd().parent / 'data' / 'raw'

# Buscar un audio de prueba (preferir Lipaugus, sino cualquiera)
preferred = data_dir / 'Lipaugus_vociferans'
if preferred.exists():
    test_file = next(preferred.glob('*.mp3'), None)
else:
    test_file = None

# Si no encontramos Lipaugus, buscar cualquier MP3 o WAV
if test_file is None:
    for ext in ('*.mp3', '*.wav'):
        files = list(data_dir.rglob(ext))
        if files:
            test_file = files[0]
            break

if test_file is None:
    raise FileNotFoundError(f'No se encontraron audios en {data_dir}')

# Cargar con nuestra función load_audio()
audio = load_audio(test_file)
print(summarize(audio))

# Escuchar el audio en el notebook
print('\n🔊 Reproduce el audio aquí abajo:')
ipd.Audio(audio.waveform, rate=audio.sample_rate)

---
## 1. La señal en el dominio del tiempo

### 1.1 ¿Qué es una señal de audio?

Un audio digital es una secuencia de números que representan la **presión del aire**
medida por un micrófono a intervalos regulares. Cada número es una **muestra** (*sample*).

Matemáticamente, una señal discreta es:

$$x[n], \quad n = 0, 1, 2, \ldots, N-1$$

donde:
- $x[n]$ es la amplitud en el instante $n$
- $N$ es el número total de muestras
- La frecuencia de muestreo $f_s$ (sample rate) nos dice cuántas muestras hay por segundo

**En nuestro proyecto:** $f_s = 22{,}050$ Hz → 22,050 muestras por segundo.

### 1.2 ¿Qué podemos ver en el dominio del tiempo?

La forma de onda nos muestra:
- **Amplitud** → qué tan fuerte es el sonido
- **Envolvente** → cómo varía la intensidad (e.g., un canto que sube y baja)
- **Silencios** → regiones donde $|x[n]| \approx 0$

Pero **NO podemos ver** qué frecuencias componen el sonido. Ahí es donde entra la STFT.

In [ ]:
# === Visualizar la forma de onda ===
fig, ax = plt.subplots(figsize=(14, 4))

# Eje de tiempo en segundos
t = np.arange(len(audio.waveform)) / audio.sample_rate

ax.plot(t, audio.waveform, color=COLORS['waveform'], linewidth=0.5, alpha=0.8)
ax.set_xlabel('Tiempo (s)')
ax.set_ylabel('Amplitud')
ax.set_title(f'Forma de onda — {Path(audio.source_path).parent.name}', fontsize=13)
ax.set_xlim(0, audio.duration_s)
ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'📊 Total de muestras: {len(audio.waveform):,}')
print(f'📊 Duración: {audio.duration_s:.2f} s')
print(f'📊 Rango de amplitud: [{audio.waveform.min():.4f}, {audio.waveform.max():.4f}]')

---
## 2. STFT — Short-Time Fourier Transform

### 2.1 El problema: el audio cambia con el tiempo

La **Transformada de Fourier (DFT)** descompone una señal en sus frecuencias componentes.
Pero tiene un problema fundamental: analiza **toda la señal de una vez**, como si las
frecuencias fueran constantes. El canto de un ave **cambia momento a momento** — primero
sube, luego baja, tiene trinos rápidos, pausas, etc.

Necesitamos saber **qué frecuencias suenan EN CADA INSTANTE**, no cuáles suenan
"en promedio" en todo el audio.

### 2.2 La solución: ventanear y transformar

La **STFT** resuelve esto de una forma elegante:

1. **Toma una ventana corta** del audio (e.g., 23 ms = 512 muestras a 22,050 Hz)
2. **Aplica la DFT** a esa ventana → obtiene las frecuencias presentes en ese instante
3. **Desliza la ventana** un paso (*hop*) hacia adelante
4. **Repite** hasta cubrir todo el audio

### 2.3 Matemáticas de la STFT

La STFT de una señal $x[n]$ se define como:

$$\text{STFT}\{x[n]\}(m, k) = X(m, k) = \sum_{n=0}^{N-1} x[n] \cdot w[n - mH] \cdot e^{-j \frac{2\pi k n}{N}}$$

donde:
- $m$ = índice del **frame temporal** (cuál ventana estamos analizando)
- $k$ = índice de **frecuencia** (cuál bin frecuencial estamos midiendo)
- $w[n]$ = **función ventana** (e.g., Hann) que suaviza los bordes del segmento
- $N$ = **tamaño de la ventana** (llamado `n_fft` en librosa). Define la resolución frecuencial.
- $H$ = **hop length** = cuántas muestras avanza la ventana entre frames

### 2.4 Los 3 hiperparámetros clave

| Parámetro | Nombre en librosa | Valor típico | ¿Qué controla? |
|-----------|------------------|--------------|----------------|
| $N$ | `n_fft` | 2048 | **Resolución frecuencial**: más grande = más detalle en frecuencia, menos en tiempo |
| $H$ | `hop_length` | 512 | **Resolución temporal**: más pequeño = más frames por segundo, más detalle temporal |
| $w[n]$ | `window` | 'hann' | **Forma de la ventana**: controla las fugas espectrales (spectral leakage) |

### 2.5 El principio de incertidumbre tiempo-frecuencia

Hay un **trade-off fundamental** (análogo al principio de Heisenberg en física cuántica):

$$\Delta t \cdot \Delta f \geq \frac{1}{4\pi}$$

- **Ventana larga** ($N$ grande): excelente resolución en frecuencia ($\Delta f$ pequeño), 
  pero mala en tiempo ($\Delta t$ grande). Ves las frecuencias con precisión, pero no sabes 
  exactamente *cuándo* sonaron.
- **Ventana corta** ($N$ pequeño): excelente resolución temporal, pero mala frecuencial.

Para bioacústica de aves, `n_fft=2048` con `hop_length=512` es un buen compromiso:
- Resolución frecuencial: $\Delta f = f_s / N = 22050/2048 \approx 10.8$ Hz (suficiente para 
  distinguir tonos de aves)
- Resolución temporal: $\Delta t = H / f_s = 512/22050 \approx 23$ ms (suficiente para 
  capturar trinos rápidos)

### 2.6 ¿Por qué la ventana de Hann?

La ventana de Hann tiene la forma:

$$w[n] = 0.5 \left(1 - \cos\left(\frac{2\pi n}{N-1}\right)\right)$$

Sin ventana (ventana rectangular), los bordes del segmento se cortan abruptamente,
creando **frecuencias fantasma** (spectral leakage). La ventana Hann suaviza los bordes
→ reduce el leakage → espectrograma más limpio. Es el estándar en procesamiento de audio.

In [ ]:
# === Visualización interactiva de la STFT ===

# 1. Calcular la STFT
n_fft = 2048      # Tamaño de ventana (resolución frecuencial)
hop_length = 512   # Paso entre ventanas (resolución temporal)

# librosa.stft retorna una matriz compleja de shape (1 + n_fft/2, T)
# donde T = número de frames temporales
D = librosa.stft(audio.waveform, n_fft=n_fft, hop_length=hop_length)

print(f'Shape de la STFT: {D.shape}')
print(f'  → {D.shape[0]} bins de frecuencia (0 a {SAMPLE_RATE//2} Hz)')
print(f'  → {D.shape[1]} frames temporales')
print(f'  → Resolución frecuencial: Δf = {SAMPLE_RATE/n_fft:.1f} Hz')
print(f'  → Resolución temporal: Δt = {hop_length/SAMPLE_RATE*1000:.1f} ms')
print(f'  → dtype: {D.dtype} (complejo — tiene magnitud y fase)')

In [ ]:
# 2. Espectrograma de magnitud (lo que se visualiza)
# |STFT|² es la densidad espectral de potencia (power spectrogram)
# Aplicamos log (en dB) para comprimir el rango dinámico:
# los sonidos suaves serían invisibles sin esta compresión.

S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Panel superior: forma de onda
t = np.arange(len(audio.waveform)) / audio.sample_rate
axes[0].plot(t, audio.waveform, color=COLORS['waveform'], linewidth=0.5)
axes[0].set_ylabel('Amplitud')
axes[0].set_title(f'Señal temporal vs Espectrograma STFT — {Path(audio.source_path).parent.name}',
                  fontsize=13)
axes[0].set_xlim(0, audio.duration_s)
axes[0].grid(True, alpha=0.3)

# Panel inferior: espectrograma
img = librosa.display.specshow(
    S_db, sr=SAMPLE_RATE, hop_length=hop_length,
    x_axis='time', y_axis='hz',
    ax=axes[1], cmap=COLORS['spectrogram']
)
axes[1].set_ylabel('Frecuencia (Hz)')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylim(0, 11000)  # Limitamos a 11 kHz (rango biológico relevante)
fig.colorbar(img, ax=axes[1], format='%+2.0f dB', label='Potencia (dB)')

plt.tight_layout()
plt.show()

print('\n💡 Observa cómo el espectrograma muestra las frecuencias del canto del ave')
print('   en función del tiempo. Los colores claros = más energía en esa frecuencia.')
print('   Esto es lo que la STFT nos permite ver y que la forma de onda NO.')

In [ ]:
# 3. Demostrar el efecto del tamaño de ventana (n_fft)
# Este es el trade-off tiempo-frecuencia en acción

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, nfft in enumerate([512, 2048, 8192]):
    D_tmp = librosa.stft(audio.waveform, n_fft=nfft, hop_length=hop_length)
    S_tmp = librosa.amplitude_to_db(np.abs(D_tmp), ref=np.max)
    librosa.display.specshow(
        S_tmp, sr=SAMPLE_RATE, hop_length=hop_length,
        x_axis='time', y_axis='hz', ax=axes[i], cmap=COLORS['spectrogram']
    )
    axes[i].set_ylim(0, 11000)
    delta_f = SAMPLE_RATE / nfft
    delta_t = nfft / SAMPLE_RATE * 1000
    axes[i].set_title(f'n_fft={nfft}\nΔf={delta_f:.1f} Hz, Δt={delta_t:.1f} ms', fontsize=10)
    if i > 0:
        axes[i].set_ylabel('')

plt.suptitle('Trade-off resolución tiempo ↔ frecuencia (principio de incertidumbre)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('⬅️  n_fft=512: buena resolución temporal, mala frecuencial (bandas gruesas)')
print('➡️  n_fft=8192: excelente resolución frecuencial, mala temporal (bordes difusos)')
print('🎯 n_fft=2048: compromiso ideal para bioacústica')

---
## 3. Espectrograma Mel

### 3.1 El problema de la escala lineal de frecuencias

El espectrograma STFT tiene un eje de frecuencias **lineal**: cada bin tiene el mismo
ancho ($\Delta f \approx 10.8$ Hz). Pero la percepción auditiva **no es lineal**:

- La diferencia entre 100 Hz y 200 Hz suena **enorme** (una octava completa)
- La diferencia entre 8000 Hz y 8100 Hz suena **imperceptible**

En otras palabras, el oído (tanto humano como de aves) es más sensible a las
**frecuencias bajas** y comprime las frecuencias altas. Esto se llama percepción
**logarítmica** de la frecuencia.

### 3.2 La escala Mel: imitando la percepción auditiva

La escala **Mel** (*Melody*) fue propuesta por Stevens, Volkmann y Newman en 1937.
Transforma frecuencias lineales (Hz) a una escala que refleja cómo percibimos el tono:

$$m = 2595 \cdot \log_{10}\left(1 + \frac{f}{700}\right)$$

Y la inversa:

$$f = 700 \cdot \left(10^{m/2595} - 1\right)$$

**Efecto práctico:** en la escala Mel, las bandas de frecuencia son:
- **Estrechas en frecuencias bajas** (donde el oído distingue bien)
- **Anchas en frecuencias altas** (donde el oído comprime)

### 3.3 ¿Por qué Mel es ideal para bioacústica?

Las vocalizaciones de aves tienen su energía principal en frecuencias **bajas-medias**
(500 Hz — 8 kHz). La escala Mel da más resolución justamente en ese rango,
permitiendo al modelo distinguir mejor las sutilezas del canto.

### 3.4 El Mel filterbank: cómo se construye

Para pasar del espectrograma STFT al espectrograma Mel:

1. Se crean $M$ filtros triangulares espaciados uniformemente en la escala Mel
   (nosotros usaremos $M = 128$ bandas)
2. Cada filtro cubre un rango de frecuencias y acumula la energía en ese rango
3. Se multiplica la matriz de filtros por el espectrograma de potencia:

$$S_{\text{mel}}(m, b) = \sum_{k=0}^{K-1} |X(m, k)|^2 \cdot H_b(k)$$

donde $H_b(k)$ es el $b$-ésimo filtro triangular Mel evaluado en el bin $k$.

En forma matricial (más eficiente computacionalmente):

$$\mathbf{S}_{\text{mel}} = \mathbf{H} \cdot |\mathbf{X}|^2$$

donde $\mathbf{H}$ es la matriz de filtros Mel de shape $(M, 1 + N/2)$.

Finalmente, se aplica logaritmo para comprimir el rango dinámico:

$$\mathbf{S}_{\text{mel, dB}} = 10 \cdot \log_{10}(\mathbf{S}_{\text{mel}} + \epsilon)$$

donde $\epsilon$ es un valor pequeño para evitar $\log(0)$.

### 3.5 Parámetros que usaremos en SelvaSonic

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `n_mels` | 128 | Estándar en clasificación de audio. 128 bandas dan suficiente detalle sin ser excesivo |
| `n_fft` | 2048 | Mismo que STFT — resolución frecuencial adecuada para cantos de aves |
| `hop_length` | 512 | ~23 ms entre frames — captura la dinámica temporal de los trinos |
| `fmin` | 0 Hz | Incluir frecuencias bajas (algunas especies amazónicas cantan bajo) |
| `fmax` | 11025 Hz | = sr/2, frecuencia máxima representable (Nyquist) |

In [ ]:
# === Visualizar la escala Mel vs lineal ===
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Izquierda: función Hz → Mel
f_hz = np.linspace(0, 11025, 1000)
f_mel = 2595 * np.log10(1 + f_hz / 700)

axes[0].plot(f_hz, f_mel, color=COLORS['accent'], linewidth=2)
axes[0].set_xlabel('Frecuencia (Hz)')
axes[0].set_ylabel('Frecuencia (Mel)')
axes[0].set_title('Conversión Hz → Mel')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=1000, color='gray', linestyle='--', alpha=0.5, label='1000 Hz = 1000 Mel')
axes[0].legend()

# Derecha: los filtros Mel triangulares
mel_fb = librosa.filters.mel(sr=SAMPLE_RATE, n_fft=n_fft, n_mels=128)
# Mostrar solo los primeros 30 filtros para que se vean claros
for i in range(0, 30):
    axes[1].plot(np.linspace(0, SAMPLE_RATE/2, mel_fb.shape[1]), mel_fb[i], alpha=0.6)
axes[1].set_xlabel('Frecuencia (Hz)')
axes[1].set_ylabel('Peso del filtro')
axes[1].set_title(f'Banco de filtros Mel (primeros 30 de {mel_fb.shape[0]})')
axes[1].set_xlim(0, 5000)  # Zoom a frecuencias bajas
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Shape del banco de filtros Mel: {mel_fb.shape}')
print(f'  → {mel_fb.shape[0]} filtros × {mel_fb.shape[1]} bins de frecuencia')
print('\n💡 Observa cómo los filtros son estrechos en frecuencias bajas (más resolución)')
print('   y anchos en frecuencias altas (menos resolución). Esto imita la percepción auditiva.')

In [ ]:
# === Calcular y visualizar el espectrograma Mel ===

# librosa.feature.melspectrogram hace todo en un paso:
# STFT → |STFT|² → multiplicar por filterbank Mel
S_mel = librosa.feature.melspectrogram(
    y=audio.waveform,
    sr=SAMPLE_RATE,
    n_fft=n_fft,
    hop_length=hop_length,
    n_mels=128,     # Número de bandas Mel
    fmin=0,          # Frecuencia mínima
    fmax=11025,      # Frecuencia máxima (Nyquist)
)

# Convertir a dB (escala logarítmica)
S_mel_db = librosa.power_to_db(S_mel, ref=np.max)

# Comparar: STFT lineal vs Mel
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# STFT con eje en Hz (lineal)
librosa.display.specshow(
    S_db, sr=SAMPLE_RATE, hop_length=hop_length,
    x_axis='time', y_axis='hz', ax=axes[0], cmap=COLORS['spectrogram']
)
axes[0].set_ylim(0, 11000)
axes[0].set_title('Espectrograma STFT (frecuencia lineal)', fontsize=12)
axes[0].set_xlabel('')

# Mel-espectrograma
img_mel = librosa.display.specshow(
    S_mel_db, sr=SAMPLE_RATE, hop_length=hop_length,
    x_axis='time', y_axis='mel', ax=axes[1], cmap=COLORS['mel']
)
axes[1].set_title('Espectrograma Mel (128 bandas) — INPUT DEL MODELO CNN', fontsize=12)
fig.colorbar(img_mel, ax=axes[1], format='%+2.0f dB', label='Potencia (dB)')

plt.tight_layout()
plt.show()

print(f'\n📊 Shape del Mel-spectrograma: {S_mel_db.shape}')
print(f'   → {S_mel_db.shape[0]} bandas Mel × {S_mel_db.shape[1]} frames temporales')
print(f'   → ESTO es lo que entra al modelo CNN como una "imagen" de 1 canal')
print(f'   → Tensor PyTorch será: [batch_size, 1, {S_mel_db.shape[0]}, {S_mel_db.shape[1]}]')

---
## 4. MFCCs — Mel-Frequency Cepstral Coefficients

### 4.1 ¿Qué son los MFCCs?

Los MFCCs son una **compresión adicional** del espectrograma Mel. Mientras el
Mel-spectrograma tiene 128 bandas, los MFCCs típicamente se reducen a solo
**13-20 coeficientes** que capturan la "forma" general del espectro.

### 4.2 ¿Cómo se calculan?

Se aplica la **Transformada Discreta del Coseno (DCT)** al log-Mel-spectrograma:

$$c_i(m) = \sum_{b=0}^{B-1} \log\left(S_{\text{mel}}(m, b)\right) \cdot \cos\left(\frac{\pi i (2b + 1)}{2B}\right)$$

donde:
- $c_i(m)$ = el $i$-ésimo coeficiente MFCC en el frame $m$
- $B$ = número de bandas Mel (128 en nuestro caso)
- La DCT es similar a la DFT pero opera con funciones coseno (reales, no complejas)

### 4.3 Interpretación intuitiva

Piensa en los MFCCs como una **descripción compacta de la forma del espectro**:

- **MFCC 0** (c₀): energía total del frame (volume general)
- **MFCC 1** (c₁): inclinación del espectro (¿más energía en bajos o en agudos?)
- **MFCC 2** (c₂): curvatura del espectro
- **MFCC 3+**: detalles cada vez más finos de la forma espectral

Es como describir una montaña: c₁ te dice la pendiente general, c₂ si tiene pico
redondeado o puntiagudo, c₃ si tiene mesetas intermedias...

### 4.4 ¿Por qué NO serán nuestro input principal?

En SelvaSonic usamos **Mel-spectrogramas** como input del CNN, **no MFCCs**. Razones:

1. **Los CNNs aprenden sus propias features**: un CNN puede extraer información
   equivalente a los MFCCs (y más) directamente del Mel-spectrograma. Los MFCCs
   son features "hechas a mano" que pre-comprimen la información.

2. **Los MFCCs descartan información**: al quedarnos solo con 13 coeficientes de 128
   bandas, perdemos el 90% de la información. Para un modelo profundo, esto puede
   limitar lo que aprende.

3. **Tendencia moderna**: desde 2017-2018, la comunidad de audio ML migró de
   MFCCs a Mel-spectrogramas. Los MFCCs siguen siendo útiles como features
   complementarios o en modelos clásicos (SVM, Random Forest).

En nuestro pipeline, los MFCCs podrían usarse como **features adicionales**
concatenados al output del CNN si necesitamos mejorar performance.

In [ ]:
# === Calcular y visualizar MFCCs ===

# Extraer 13 MFCCs (estándar en speech/audio processing)
mfccs = librosa.feature.mfcc(
    y=audio.waveform,
    sr=SAMPLE_RATE,
    n_mfcc=13,        # Número de coeficientes
    n_fft=n_fft,
    hop_length=hop_length,
    n_mels=128,
)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Panel 1: Forma de onda
t = np.arange(len(audio.waveform)) / SAMPLE_RATE
axes[0].plot(t, audio.waveform, color=COLORS['waveform'], linewidth=0.5)
axes[0].set_ylabel('Amplitud')
axes[0].set_title('Señal temporal', fontsize=11)
axes[0].set_xlim(0, audio.duration_s)
axes[0].grid(True, alpha=0.3)

# Panel 2: Mel-spectrograma
librosa.display.specshow(
    S_mel_db, sr=SAMPLE_RATE, hop_length=hop_length,
    x_axis='time', y_axis='mel', ax=axes[1], cmap=COLORS['mel']
)
axes[1].set_title('Espectrograma Mel (128 bandas)', fontsize=11)
axes[1].set_xlabel('')

# Panel 3: MFCCs
img_mfcc = librosa.display.specshow(
    mfccs, sr=SAMPLE_RATE, hop_length=hop_length,
    x_axis='time', ax=axes[2], cmap='coolwarm'
)
axes[2].set_ylabel('Coeficiente MFCC')
axes[2].set_title('MFCCs (13 coeficientes)', fontsize=11)
fig.colorbar(img_mfcc, ax=axes[2], label='Valor')

plt.tight_layout()
plt.show()

print(f'📊 Shape de MFCCs: {mfccs.shape}')
print(f'   → {mfccs.shape[0]} coeficientes × {mfccs.shape[1]} frames temporales')
print(f'\n📊 Compresión: Mel tiene {S_mel_db.shape[0]} bandas → MFCC reduce a {mfccs.shape[0]} coeficientes')
print(f'   → Factor de compresión: {S_mel_db.shape[0]/mfccs.shape[0]:.1f}x')

---
## 5. Resumen: la cadena completa

### 5.1 Pipeline de representación de audio en SelvaSonic

```
Audio crudo x[n]           Shape: (N,)          1D — amplitud vs tiempo
     │
     ▼ STFT (ventana Hann, n_fft=2048)
     │
Espectrograma complejo     Shape: (1025, T)     2D — complejo
     │
     ▼ |·|² (magnitud al cuadrado)
     │
Power spectrogram          Shape: (1025, T)     2D — real, escala lineal Hz
     │
     ▼ Mel filterbank (128 filtros triangulares)
     │
Mel spectrogram            Shape: (128, T)      2D — real, escala Mel
     │
     ▼ log₁₀(·)  +  normalización
     │
Log-Mel spectrogram   ──→  INPUT DEL MODELO CNN  ← usamos este
     │
     ▼ DCT (13 coeficientes)
     │
MFCCs                      Shape: (13, T)       2D — features compactos
```

### 5.2 ¿Qué usaremos en el modelo?

| Representación | Rol en SelvaSonic |
|----------------|------------------|
| Log-Mel spectrograma (128 × T) | **Input principal** del CNN. Se trata como una imagen de 1 canal |
| MFCCs (13 × T) | **Opcional** — features complementarios si se necesitan |

### 5.3 Conexión con la Semana 2

En la Semana 2 vamos a:
1. Segmentar los audios en clips de **5 segundos** → cada clip produce un Mel-spectrograma de shape (128, ~216)
2. Guardar los espectrogramas como tensores `.pt` en `data/processed/`
3. Crear la clase `AmazonAudioDataset` que alimente los espectrogramas al DataLoader de PyTorch

La base teórica que aprendimos aquí es la que justifica **cada parámetro** que usaremos en esa extracción.

In [ ]:
# === Comparación final de las 3 representaciones ===

# Tomar solo los primeros 3 segundos para mayor claridad visual
clip = load_audio(test_file, duration=3.0)

# Calcular las 3 representaciones
D_clip = librosa.stft(clip.waveform, n_fft=n_fft, hop_length=hop_length)
S_clip_db = librosa.amplitude_to_db(np.abs(D_clip), ref=np.max)

S_mel_clip = librosa.feature.melspectrogram(
    y=clip.waveform, sr=SAMPLE_RATE, n_fft=n_fft,
    hop_length=hop_length, n_mels=128
)
S_mel_clip_db = librosa.power_to_db(S_mel_clip, ref=np.max)

mfccs_clip = librosa.feature.mfcc(
    y=clip.waveform, sr=SAMPLE_RATE, n_mfcc=13,
    n_fft=n_fft, hop_length=hop_length, n_mels=128
)

# Visualizar las 4 representaciones lado a lado
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

# 1. Forma de onda
t_clip = np.arange(len(clip.waveform)) / SAMPLE_RATE
axes[0].plot(t_clip, clip.waveform, color=COLORS['waveform'], linewidth=0.5)
axes[0].set_ylabel('Amplitud')
axes[0].set_title('1. Señal temporal x[n]', fontsize=11)
axes[0].set_xlim(0, 3.0)
axes[0].grid(True, alpha=0.3)

# 2. STFT
librosa.display.specshow(S_clip_db, sr=SAMPLE_RATE, hop_length=hop_length,
                         x_axis='time', y_axis='hz', ax=axes[1], cmap=COLORS['spectrogram'])
axes[1].set_ylim(0, 11000)
axes[1].set_title(f'2. Espectrograma STFT — shape {S_clip_db.shape}', fontsize=11)
axes[1].set_xlabel('')

# 3. Mel
librosa.display.specshow(S_mel_clip_db, sr=SAMPLE_RATE, hop_length=hop_length,
                         x_axis='time', y_axis='mel', ax=axes[2], cmap=COLORS['mel'])
axes[2].set_title(f'3. Espectrograma Mel — shape {S_mel_clip_db.shape} ⭐ INPUT DEL MODELO', fontsize=11)
axes[2].set_xlabel('')

# 4. MFCCs
librosa.display.specshow(mfccs_clip, sr=SAMPLE_RATE, hop_length=hop_length,
                         x_axis='time', ax=axes[3], cmap='coolwarm')
axes[3].set_ylabel('Coeficiente')
axes[3].set_title(f'4. MFCCs — shape {mfccs_clip.shape}', fontsize=11)

plt.tight_layout()
plt.show()

print('\n🎯 RESUMEN DE SHAPES (para 3 segundos de audio):')
print(f'   Audio crudo:      {clip.waveform.shape}  (1D)')
print(f'   STFT:             {S_clip_db.shape}  (mucha información, escala lineal)')
print(f'   Mel-spectrograma: {S_mel_clip_db.shape}  (128 bandas, escala perceptual) ⭐')
print(f'   MFCCs:            {mfccs_clip.shape}   (compresión extrema)')